# Sovereign AI That Understands Your Network

**The one idea:** an AI security assistant needs two very different kinds of knowledge, and they belong in two different places.

- **Security Knowledge (SK)** — *how to reason about vulnerabilities* (what a weakness is, its CWE, how to remediate). This belongs in the **model's weights**.

- **Customer Knowledge (CK)** — *your* network and estate: assets, policies, incidents, SLAs. This is private, changes often, and belongs in **RAG** (a retrieval layer), never baked into weights.

Everything here runs **locally on a laptop** via Ollama (Apple Silicon) — no cloud, no data leaving the machine, on **infrastructure you control**.

**How this notebook works.** We build four stages of an on-prem assistant, then **evaluate all four across 32 vulnerability-triage scenarios** for a synthetic company, *LatamPay*. Each stage section below only **assembles that stage's artifact**. The **testing and conclusions live in the final section**, where every stage is scored on **SK** (correct weakness) and **CK** (customer facts grounded).

| Stage | Artifact it builds | Knowledge added |
| --- | --- | --- |
| **1** | Llama 3.1 8B, no retrieval | Generic model weights only (baseline) |
| **2** | Llama 3.1 8B + CK-RAG | Your customer facts, retrieved at query time |
| **3** | SK-LoRA + CK-RAG | Build-your-own security knowledge in the weights |
| **4** | Foundation-Sec + CK-RAG | A purpose-built security base model |

**Note:** *LatamPay is fully synthetic; the customer pack contains no real data. All models are open-weight and run offline.*


## 1. Environment Sanity and Library Imports

Confirm the lab can see its data and helper scripts, load the shared helpers (the same ones used by the batch benchmark, so results are consistent), and load the 32 scenarios.


In [25]:
from pathlib import Path
from importlib import import_module
from importlib import util as importlib_util
import json
import subprocess
import sys

display_mod = import_module('IPython.display')
Markdown = display_mod.Markdown
display = display_mod.display

# Self-contained lab: everything resolves from the repo root (the notebook's working dir).
LAB_ROOT = Path.cwd()
DATA_DIR = LAB_ROOT / 'data'
CK_DB = LAB_ROOT / 'customer_ck_chromadb'

sys.path.insert(0, str(DATA_DIR))

print('Path check:')
for name, p in {
    'LAB_ROOT': LAB_ROOT,
    'DATA_DIR': DATA_DIR,
    'CK_DB (RAG store)': CK_DB,
    'scenarios.json': DATA_DIR / 'scenarios.json',
}.items():
    print(f'  {name:26} exists={p.exists()}')


Path check:
  LAB_ROOT                   exists=True
  DATA_DIR                   exists=True
  CK_DB (RAG store)          exists=True
  scenarios.json             exists=True


In [26]:
# Load the shared helpers (prompting, retrieval, scoring) from the eval scripts.
def load_module(module_name, file_path):
    spec = importlib_util.spec_from_file_location(module_name, file_path)
    module = importlib_util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

eval_phase12 = load_module('eval_phase12', DATA_DIR / 'eval_phase12.py')
eval_stage4 = load_module('eval_stage4', DATA_DIR / 'eval_stage4.py')
eval_stage3 = load_module('eval_stage3', DATA_DIR / 'eval_stage3.py')

DEFAULT_COLLECTION = eval_phase12.DEFAULT_COLLECTION
DEFAULT_EMBED_MODEL = eval_phase12.DEFAULT_EMBED_MODEL
DEFAULT_OLLAMA_URL = eval_phase12.DEFAULT_OLLAMA_URL
STAGE1_SYSTEM = eval_phase12.STAGE1_SYSTEM
STAGE2_SYSTEM = eval_phase12.STAGE2_SYSTEM
STAGE3_SYSTEM = eval_stage3.STAGE3_SYSTEM
STAGE4_SYSTEM = eval_stage4.STAGE4_SYSTEM
heuristic_score = eval_phase12.heuristic_score
load_retriever = eval_phase12.load_retriever
load_scenarios = eval_phase12.load_scenarios
ollama_generate = eval_phase12.ollama_generate
retrieve_ck = eval_phase12.retrieve_ck
scenario_prompt = eval_phase12.scenario_prompt

# Decoding held constant across stages so only the model/knowledge differs.
OLLAMA_URL = DEFAULT_OLLAMA_URL
NUM_PREDICT = 180
NUM_THREAD = 16
TEMPERATURE = 0.0   # deterministic: same answers every run
TIMEOUT = 300

# Each stage will register its artifact here (built in sections 2-5).
STAGES = {}
STAGE_ORDER = ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']

print('Shared helpers loaded. Decoding: temperature=0 (deterministic).')


Shared helpers loaded. Decoding: temperature=0 (deterministic).


In [27]:
# Helper functions: retrieval, scoring guardrails, one run_stage(), and the batch benchmark.
retriever_cache = {'embedder': None, 'collection': None}

def get_retriever():
    if retriever_cache['embedder'] is None:
        embedder, collection = load_retriever(str(CK_DB), DEFAULT_COLLECTION, DEFAULT_EMBED_MODEL, 'cpu')
        retriever_cache['embedder'] = embedder
        retriever_cache['collection'] = collection
    return retriever_cache['embedder'], retriever_cache['collection']

def get_ck_context(scenario, k=2, policy_k=2):
    embedder, collection = get_retriever()
    return retrieve_ck(scenario, embedder, collection, k=k, policy_k=policy_k)

def _has_metadata_blast_radius(text):
    t = text.lower()
    return any(x in t for x in ['metadata endpoint', '169.254.169.254', 'cloud credentials',
                                'cloud instance metadata', 'instance metadata'])

def _has_critical_business_context(text):
    t = text.lower()
    return any(x in t for x in ['tier-0', 'pci', '24h', '24 hours', 'sev-1', 'payment-gateway'])

def _apply_ck_guardrail(score, ck_context, response_text):
    if _has_metadata_blast_radius(ck_context) and _has_metadata_blast_radius(response_text):
        score['ck_term_hit'] = True
    if _has_critical_business_context(ck_context) and _has_critical_business_context(response_text):
        score['ck_term_hit'] = True
    return score

def _apply_dependency_guardrail(score, uses_ck):
    if not uses_ck:            # no CK, so CK/applicability cannot be credited
        score['ck_term_hit'] = False
        score['applicability_hit'] = False
    elif not score['sk_hit']:  # applicability only counts if the weakness was correctly identified
        score['applicability_hit'] = False
    return score

def run_stage(stage_name, scenario, k=2, policy_k=2):
    config = STAGES[stage_name]
    retrieved, ck_context = [], ''
    if config['ck']:
        ck_context, retrieved = get_ck_context(scenario, k=k, policy_k=policy_k)
        prompt = scenario_prompt(scenario, ck_context)
    else:
        prompt = scenario_prompt(scenario)
    response_text, elapsed = ollama_generate(
        model=config['model'], system=config['system'], prompt=prompt,
        ollama_url=OLLAMA_URL, num_predict=NUM_PREDICT, num_thread=NUM_THREAD,
        temperature=TEMPERATURE, timeout=TIMEOUT,
    )
    score = heuristic_score(stage_name.lower().replace(' ', ''), scenario, response_text)
    score = _apply_ck_guardrail(score, ck_context, response_text)
    score = _apply_dependency_guardrail(score, config['ck'])
    return {'stage': stage_name, 'model': config['model'], 'scenario_id': scenario['id'],
            'answer': response_text, 'elapsed_seconds': elapsed, 'score': score}

def markdown_table(headers, rows):
    header = '| ' + ' | '.join(headers) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(headers)) + ' |'
    body = ['| ' + ' | '.join(str(i) for i in row) + ' |' for row in rows]
    return '\n'.join([header, sep, *body])

def run_benchmark(scenarios, stage_names, verbose=True):
    """Evaluate every stage across every scenario; return per-stage tallies + raw records.

    Stage-major order: run one model over all scenarios before switching models, so
    Ollama keeps each model resident (~2 swaps total) instead of reloading weights on
    every call. Same tallies as any order, but much faster.
    """
    tally = {s: {'sk': 0, 'ck': 0, 'app': 0, 't': 0.0, 'n': 0} for s in stage_names}
    records = []
    total = len(scenarios)
    for stage in stage_names:
        if verbose:
            ck_flag = '+ CK-RAG' if STAGES[stage]['ck'] else 'no CK'
            print(f'=== {stage}: {STAGES[stage]["model"]} ({ck_flag}) ===')
        for i, sc in enumerate(scenarios, 1):
            r = run_stage(stage, sc)
            s = r['score']
            tally[stage]['sk'] += int(bool(s['sk_hit']))
            tally[stage]['ck'] += int(bool(s['ck_term_hit']))
            tally[stage]['app'] += int(bool(s['applicability_hit']))
            tally[stage]['t'] += r['elapsed_seconds']
            tally[stage]['n'] += 1
            records.append(r)
            if verbose:
                print(f'  [{i:2}/{total}] {sc["id"]:34} SK={int(bool(s["sk_hit"]))} CK={int(bool(s["ck_term_hit"]))}')
    return tally, records

print('Helpers ready: run_stage(), run_benchmark(), get_ck_context().')


Helpers ready: run_stage(), run_benchmark(), get_ck_context().


In [28]:
# Confirm the three Ollama models are present before running the benchmark.
def run_command(command):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    return result

print('Ollama models available:\n')
ollama_list = run_command('ollama list')
required = ['llama3.1:8b-instruct-q4_K_M', 'llama3.1-8b-sk-lora', 'foundation-sec-8b-q4']
missing = [t for t in required if t not in ollama_list.stdout]
print('\nRequired tags missing:', missing if missing else 'none — all present')


Ollama models available:

NAME                             ID              SIZE      MODIFIED     
sk-demo-foundation-sec:latest    c4224d247b70    4.9 GB    5 days ago      
sk-demo-lora:latest              a9e11d87a1b2    4.9 GB    5 days ago      
sk-demo-base:latest              913f9dbe4a66    4.9 GB    5 days ago      
foundation-sec-8b-q4:latest      c65c8ff25197    4.9 GB    6 days ago      
llama3.1-8b-sk-lora:latest       e1bae939ed9f    4.9 GB    6 days ago      
llama3.1:8b-instruct-q4_K_M      46e0c10c039e    4.9 GB    6 days ago      
nomic-embed-text:latest          0a109f422b47    274 MB    4 weeks ago     
qwen3:8b                         500a1f067a9f    5.2 GB    8 weeks ago     
qwen2.5:3b                       357c53fb659c    1.9 GB    2 months ago

Required tags missing: none — all present


In [29]:
# Load the 32 benchmark scenarios (the test set for the final section).
benchmark_scenarios = load_scenarios(DATA_DIR / 'scenarios.json')
print(f'Loaded {len(benchmark_scenarios)} benchmark scenarios.')
archetypes = {}
for sc in benchmark_scenarios:
    archetypes[sc['archetype']] = archetypes.get(sc['archetype'], 0) + 1
print('By archetype:')
for a, n in sorted(archetypes.items()):
    print(f'  {a:22} {n}')


Loaded 32 benchmark scenarios.
By archetype:
  hit                    16
  miss                   8
  sk_gated               8


## 2. Stage 1 — Llama 3.1 alone (baseline artifact)

A **stock, general-purpose** open model (Meta's Llama 3.1 8B, four bits quantization) with **no customer context** — it triages using only what is in its weights.

This is the **baseline**: it knows *general* security concepts but has **no idea who LatamPay is**, so it cannot judge whether *you* are affected or how urgent it is. The cell below only registers the artifact; no test runs yet.


In [30]:
STAGES['Stage 1'] = {'model': 'llama3.1:8b-instruct-q4_K_M', 'system': STAGE1_SYSTEM, 'ck': False}
print('Stage 1 artifact registered:')
print('  model :', STAGES['Stage 1']['model'])
print('  CK-RAG:', STAGES['Stage 1']['ck'])


Stage 1 artifact registered:
  model : llama3.1:8b-instruct-q4_K_M
  CK-RAG: False


## 3. Stage 2 — Llama 3.1 + Customer Knowledge (RAG artifact)

Same model, same weights — we only change **what we put in the prompt**: LatamPay's own facts, retrieved from a local vector database.

**How we built the CK database (once, offline):**
1. **Collected** LatamPay's documents — asset inventory, security policy, incident history, SBOM, network zones, data classification.
2. **Chunked** them into passages (markdown split by heading; the SBOM into one passage per component) so related facts stay together.
3. **Embedded** each passage with a local model (BGE-small) and **stored** the vectors in a local **Chroma** database on disk — no cloud, nothing leaves the machine.

At query time we embed the scenario, pull the nearest passages, and paste them into the prompt. The cell below registers the Stage 2 artifact, builds the retriever, and shows one example retrieval.


In [31]:
STAGES['Stage 2'] = {'model': 'llama3.1:8b-instruct-q4_K_M', 'system': STAGE2_SYSTEM, 'ck': True}

# Build the retriever artifact (loads the embedding model + CK vector store).
example = benchmark_scenarios[0]
ck_context, retrieved = get_ck_context(example)

print('Stage 2 artifact registered:')
print('  model :', STAGES['Stage 2']['model'])
print('  CK-RAG:', STAGES['Stage 2']['ck'])
print(f'\nExample retrieval for {example["id"]} — snippets fed to the model (raw text):\n')
# Print as plain text (not rendered Markdown) so it is clearly a retrieval output.
print(ck_context[:1000])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24318.70it/s]

Stage 2 artifact registered:
  model : llama3.1:8b-instruct-q4_K_M
  CK-RAG: True

Example retrieval for S01-struts-ognl-hit — snippets fed to the model (raw text):

[1] type=scenario source=sbom.cdx.json
SBOM component: Apache Struts 2.5.33 (framework). Web framework used by payment-gateway. Uses OGNL expression evaluation.

[2] type=scenario source=incident_tickets.md
# LatamPay S.A. — Prior Incident Tickets (History)
## INC-2023-1102 — Apache Struts exploitation attempt on payment-gateway
- **Date:** 2023-11
- **Summary:** WAF blocked crafted OGNL expression payloads aimed at the **payment-gateway** Struts endpoints. No compromise.
- **Lessons:** Struts/OGNL expression-injection weaknesses on the payment-gateway are treated as **high-confidence, high-priority** because the asset is tier-0 and internet-facing. Keep Struts patched aggressively.

[3] type=policy source=security_policy.md
# LatamPay S.A. — Security Policy & Remediation SLAs
## Escalation rules
1. Any exploitable weaknes

## 4. Stage 3 — SK-LoRA + Customer Knowledge (build-your-own SK artifact)

Now we improve the **weights** — the home of Security Knowledge — while keeping the exact same RAG.

**How we built the SK-LoRA adapter (once, offline):**
1. **Built a training set** of instruction pairs from public security sources — the MITRE CWE catalog and NVD CVE descriptions, plus a small curated seed set (describe-a-weakness → CWE, and CWE → remediation).
2. **Fine-tuned a small QLoRA adapter** (low rank, ~1 epoch) on Llama-3.1-8B — only the tiny adapter is trained, not the whole model, so it is cheap and fast.
3. **Merged** the adapter into the base and **exported to GGUF (Q4)** so Ollama can run it like any other model.

The result is a sharper **weakness classifier** — the *build-your-own* path to sovereign SK, on hardware you control. The cell below registers the artifact.


In [32]:
STAGES['Stage 3'] = {'model': 'llama3.1-8b-sk-lora', 'system': STAGE3_SYSTEM, 'ck': True}
print('Stage 3 artifact registered:')
print('  model :', STAGES['Stage 3']['model'], '(Llama 3.1 8B + Security-Knowledge LoRA adapter)')
print('  CK-RAG:', STAGES['Stage 3']['ck'])


Stage 3 artifact registered:
  model : llama3.1-8b-sk-lora (Llama 3.1 8B + Security-Knowledge LoRA adapter)
  CK-RAG: True


## 5. Stage 4 — Foundation-Sec + Customer Knowledge (adopt-a-specialist artifact)

Instead of adapting a general model yourself, **adopt a base that is already a security specialist** — same CK-RAG as before.

**What is Foundation-Sec?** Cisco **Foundation-Sec-8B** is a **purpose-built, open-weight (Apache-2.0)** security foundation model on a Llama-3.1-8B backbone, pre-trained on a large security corpus. Its **weights already carry deep Security Knowledge**, so strong security reasoning comes out of the box — then you add *your* facts through the same retrieval layer.

This is the *adopt-a-specialist* path to sovereign SK. The cell below registers the artifact.


In [33]:
STAGES['Stage 4'] = {'model': 'foundation-sec-8b-q4', 'system': STAGE4_SYSTEM, 'ck': True}
print('Stage 4 artifact registered:')
print('  model :', STAGES['Stage 4']['model'], '(Cisco Foundation-Sec-8B, Apache-2.0)')
print('  CK-RAG:', STAGES['Stage 4']['ck'])
print('\nAll stages registered:', ', '.join(STAGES.keys()))


Stage 4 artifact registered:
  model : foundation-sec-8b-q4 (Cisco Foundation-Sec-8B, Apache-2.0)
  CK-RAG: True

All stages registered: Stage 1, Stage 2, Stage 3, Stage 4


## 6. Benchmark Across All 32 Scenarios, and Conclusions

This is the test. Every stage is run against **all 32 scenarios** in a single deterministic pass (`temperature = 0`). **The scenarios, the customer-knowledge prompt, and the decoding are identical across stages — only the model changes** (Stage 1 has no CK).

**How to read the columns:** **SK** = correct weakness class · **CK** = customer facts correctly grounded.

> Running all four stages over 32 scenarios is ~128 model calls and takes a few minutes on a laptop. Progress prints per scenario.


In [34]:
# Run the full 32-scenario benchmark across all four stages (this is the actual test).
tally, records = run_benchmark(benchmark_scenarios, STAGE_ORDER, verbose=True)

rows = []
for stage in STAGE_ORDER:
    d = tally[stage]
    n = d['n']
    rows.append([
        stage,
        STAGES[stage]['model'],
        f"{d['sk']}/{n}",
        f"{d['ck']}/{n}",
        f"{d['t']/n:.2f}s",
    ])

display(Markdown('### Benchmark results — all 32 scenarios (deterministic, temperature = 0)'))
display(Markdown(markdown_table(['Stage', 'Model', 'SK', 'CK grounding', 'Avg time'], rows)))


=== Stage 1: llama3.1:8b-instruct-q4_K_M (no CK) ===
  [ 1/32] S01-struts-ognl-hit                SK=0 CK=0
  [ 2/32] S02-log4j-jndi-miss                SK=1 CK=0
  [ 3/32] S03-deserialization-sk-gated       SK=1 CK=0
  [ 4/32] S04-path-traversal-hit             SK=1 CK=0
  [ 5/32] S05-ssrf-hit                       SK=1 CK=0
  [ 6/32] S06-ldap-injection-hit             SK=1 CK=0
  [ 7/32] S07-sql-injection-mysql-miss       SK=0 CK=0
  [ 8/32] S08-toctou-sk-gated                SK=0 CK=0
  [ 9/32] S09-missing-auth-hit               SK=1 CK=0
  [10/32] S10-xxe-decommissioned-miss        SK=0 CK=0
  [11/32] S11-hardcoded-creds-hit            SK=1 CK=0
  [12/32] S12-open-redirect-sk-gated         SK=1 CK=0
  [13/32] S13-sqli-postgres-payment-hit      SK=0 CK=0
  [14/32] S14-stored-xss-portal-hit          SK=1 CK=0
  [15/32] S15-file-upload-document-hit       SK=1 CK=0
  [16/32] S16-idor-portal-hit                SK=0 CK=0
  [17/32] S17-os-command-injection-payment-hit SK=1 CK=0
  [18/32] 

### Benchmark results — all 32 scenarios (deterministic, temperature = 0)

| Stage | Model | SK | CK grounding | Avg time |
| --- | --- | --- | --- | --- |
| Stage 1 | llama3.1:8b-instruct-q4_K_M | 17/32 | 0/32 | 2.22s |
| Stage 2 | llama3.1:8b-instruct-q4_K_M | 18/32 | 29/32 | 4.52s |
| Stage 3 | llama3.1-8b-sk-lora | 21/32 | 21/32 | 3.71s |
| Stage 4 | foundation-sec-8b-q4 | 25/32 | 31/32 | 5.95s |

### Conclusions

Same scenarios, same customer-knowledge prompt, same decoding — **only the model changes**.

- **Stage 1 sets the baseline.** A stock general model scores some SK from general knowledge, but with **no customer grounding at all** — it can't tell whether a weakness even applies to LatamPay's estate.
- **Stage 2 proves CK-RAG.** The moment LatamPay facts are retrieved, customer grounding jumps from zero. Customer knowledge belongs in retrieval: external, current, inspectable.
- **Stage 3 proves the build path.** A small, low-cost LoRA lifts Security Knowledge in the weights — a strong result for a day's work — while keeping all of RAG. A narrow specialist today; more investment closes the rest of the gap.
- **Stage 4 proves the specialist path.** A purpose-built, open-weight security base is strongest on both axes.

**The architecture holds throughout:** **Customer Knowledge stays external in RAG; Security Knowledge improves in the weights** — whether you adapt a model you already have, or adopt a security-specialized foundation model. And all of it runs on **infrastructure you control** — sovereign by design.
